### This Notebook generates the Input data set for RQ1

# Occupational Task Data Construction

This notebook constructs the occupational task-level dataset used for downstream GenAI task-affordance classification and analysis.

The workflow starts from the predefined `ONET_Task_Universe.csv`, which identifies the occupation–task population to be retained. It then enriches those task records with the latest O*NET task-statement and task-importance metadata while preserving task provenance fields such as `onet_soc_code` and `task_source_type`.

The final output reproduces the required structure of `Occupational_Task_Universe_ONET30_v2.csv` while retaining additional provenance fields needed for the updated occupational universe.

In [ ]:
# STEP 1
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes-full")

TASK_UNIVERSE_PATH = (
    ROOT / "01_BLS_Raw_Data_Construct" / "Output" / "ONET_Task_Universe.csv"
)

INPUT_DIR = ROOT / "02_Task_Data_and_Classification" / "Input"

TASK_RATINGS_PATH = INPUT_DIR / "Task Ratings.xlsx"

In [ ]:
# ------------------------------------------------------------
# STEP 2: Load source files
# ------------------------------------------------------------

task_universe = pd.read_csv(TASK_UNIVERSE_PATH)

task_universe = task_universe.rename(columns={
    "Incumbents Responding": "incumbents_responding",
    "Date": "statement_date",
    "Domain Source": "statement_source",
})

task_ratings = pd.read_excel(TASK_RATINGS_PATH)

print("Task universe:", task_universe.shape)
print("Task ratings:", task_ratings.shape)

In [ ]:
# ------------------------------------------------------------
# STEP 3: Extract task importance ratings
# ------------------------------------------------------------
importance_task_rating = (
    task_ratings
    .loc[task_ratings["Scale ID"].eq("IM")]
    .copy()
)

In [ ]:
# ------------------------------------------------------------
# STEP 4: Standardize task importance fields
# ------------------------------------------------------------

importance_task_rating = (
    importance_task_rating[
        [
            "O*NET-SOC Code",
            "Task ID",
            "Data Value",
            "N",
            "Standard Error",
            "Lower CI Bound",
            "Upper CI Bound",
            "Recommend Suppress",
            "Date",
            "Domain Source",
        ]
    ]
    .rename(
        columns={
            "O*NET-SOC Code": "onet_soc_code",
            "Task ID": "task_id",
            "Data Value": "task_importance",
            "N": "importance_n",
            "Standard Error": "importance_se",
            "Lower CI Bound": "importance_lower_ci",
            "Upper CI Bound": "importance_upper_ci",
            "Recommend Suppress": "importance_suppress",
            "Date": "importance_date",
            "Domain Source": "importance_source",
        }
    )
)

In [ ]:
# ------------------------------------------------------------
# STEP 5: Merge task importance ratings
# ------------------------------------------------------------

task_universe = task_universe.merge(
    importance_task_rating,
    on=["onet_soc_code", "task_id"],
    how="left",
    validate="one_to_one"
)

print("Task universe:", task_universe.shape)
print("Missing task importance:", task_universe["task_importance"].isna().sum())

In [ ]:
# ------------------------------------------------------------
# STEP6: Fill missing task importance ratings
# ------------------------------------------------------------

task_universe_fill_missing_imp_rating = task_universe.copy()

occupation_mean = (
    task_universe_fill_missing_imp_rating
    .groupby("onet_soc_code")["task_importance"]
    .transform("mean")
)

task_universe_fill_missing_imp_rating["task_importance_effective"] = (
    task_universe_fill_missing_imp_rating["task_importance"]
    .fillna(occupation_mean)
    .fillna(1.0)
)

In [ ]:
# ------------------------------------------------------------
# STEP 7:Add O*NET release
# ------------------------------------------------------------

task_universe_fill_missing_imp_rating["onet_release"] = "30.0"

In [ ]:
task_universe_fill_missing_imp_rating.to_csv(
    ROOT / "02_Task_Data_and_Classification" / "Output" / "Occupational_Task_Universe_ONET30_v2.csv",
    index=False
)